# Queueing theory for data scientists

The prediction target is not the forecasted arrival rate; it is the distribution of time jobs spend waiting. This notebook is the executable companion to [the article](article.md). It uses synthetic, stationary FCFS queues and runs a reduced experiment; the full published estimates come from `python analysis.py`.

The main variables are arrival intensity \(\lambda\), mean service \(E[S]\), utilization \(\rho=\lambda E[S]/c\), queue wait \(W_q\), and end-to-end time \(T=W_q+S\). The simulation reports queue waits **excluding** service.

## The exact baselines

For a stationary single-server M/G/1 queue with Poisson arrivals, iid finite-second-moment service, FCFS and \(\rho<1\), the Pollaczek–Khinchine formula is

\[E[W_q]=\frac{\lambda E[S^2]}{2(1-\rho)}=\frac{\lambda E[S]^2(1+c_s^2)}{2(1-\rho)}.\]

With \(\lambda=0.8\) per minute and \(E[S]=1\) minute, this is 2 minutes for deterministic service, 4 minutes for exponential service and 10 minutes for lognormal service with squared CV 4. These are theoretical long-run means, not empirical estimates.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from analysis import (
    SPECS, THRESHOLDS, expected_mg1_wait, expected_mmc_wait,
    kingman_wait, prediction, run_experiment, simulate_fcfs,
)

assert np.isclose(expected_mg1_wait(0.8, 1.0, 0.0), 2.0)
assert np.isclose(expected_mg1_wait(0.8, 1.0, 1.0), 4.0)
assert np.isclose(expected_mg1_wait(0.8, 1.0, 4.0), 10.0)
assert np.isclose(expected_mmc_wait(0.8, 1.0, 2), 4.0 / 21.0)
print('Analytic queue waits: 2.0, 4.0, 10.0, and', round(expected_mmc_wait(0.8, 1.0, 2), 3), 'minutes')

## A worked trace

Three jobs arrive at times 0, 0.5 and 1 minute. Each requires one minute of service. Under one server, waits are 0, 0.5 and 1 minute; under two servers all start immediately. The simulator implements this same FCFS mechanism at scale.

In [ ]:
arrivals = np.array([0.0, 0.5, 1.0])
service = np.ones(3)
print('One server:', simulate_fcfs(arrivals, service, servers=1))
print('Two servers:', simulate_fcfs(arrivals, service, servers=2))

## A fixed mean is not a fixed queue

Six scenarios share arrival rate 0.8 per minute. Four keep mean service at one minute and change variability or staffing. One speeds service by 15%. A two-phase hyperexponential renewal interarrival distribution has squared CV 4, but **does not** introduce autocorrelation or intraday patterns. All experiments start empty and discard warm-up arrivals; independent replications provide simulation uncertainty.

Run the small experiment below in the notebook. Do not quote its unstable tail estimates in the article: the publication tables were generated with sixteen longer replications.

In [ ]:
summary, tails, sensitivity = run_experiment(quick=True)
for row in summary:
    print(f"{row['label']:30} exact/approx={row['predicted_mean_wait_min']:5.2f}  simulation={row['simulated_mean_wait_min']:6.2f}  p95={row['simulated_p95_wait_min']:6.2f}")

## Do the analytic predictions match, and what happens in the tail?

The analytic benchmark is exact for Poisson-arrival scenarios. For the GI/M/1 arrival-variability case it is Kingman's **approximation**, not an exact identity. A wide Monte Carlo interval is not evidence that a model is false; it may simply indicate rare long waits and too few repetitions.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
labels = [str(row['scenario']) for row in summary]
x = np.arange(len(summary))
ax.scatter(x - 0.1, [row['simulated_mean_wait_min'] for row in summary], label='Simulated mean')
ax.scatter(x + 0.1, [row['predicted_mean_wait_min'] for row in summary], marker='x', label='Exact / Kingman approx.')
ax.set_xticks(x, labels)
ax.set_ylabel('Queue wait (minutes)')
ax.legend()
ax.grid(axis='y', alpha=0.2)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
for slug in ['md1', 'mm1', 'mg1', 'gim1']:
    points = [row for row in tails if row['scenario'] == slug]
    ax.plot([p['threshold_min'] for p in points], [p['fraction_exceeding'] for p in points], '-o', label=slug)
ax.set_yscale('log')
ax.set_xlabel('Threshold (minutes)')
ax.set_ylabel('Fraction exceeding threshold')
ax.legend()
ax.grid(alpha=0.2)
plt.show()

## Interpretation and limitations

A point forecast of average demand cannot distinguish the service distributions in this experiment. Queueing formulas make different assumptions and summarize different properties: utilization, the second moment of service, server count and (approximately) arrival variation. Long waits emerge from the interaction of these factors and the ordering of events.

The quick experiment is suitable only for inspecting mechanics. Full tables and figures under `results/` and `figures/` come from the reproducible command `python analysis.py`, with independent repetitions, more arrivals and a utilization sensitivity sweep. This study has no abandonment, customer redials, changing staffing, priority queues, or real organization data. See the manuscript for original research sources and the linked contact-centre studies for those more complex mechanisms.